# 11 — Necessity Hypothesis (Technical / Lifecycle): Replacing Physical PBX / 必要性假設檢定（技術／生命週期）

**EN.** This notebook applies **null-hypothesis significance testing** (per the framework at
[yongxi-stat.com/hypothesis-stat](https://www.yongxi-stat.com/hypothesis-stat/)) to the *upstream*
question that notebook 10 (financial) does **not** answer: **is replacing the physical PBX even
*necessary*?** The driver here is **technical obsolescence** — the share of the installed solution
base that is **end-of-life (EOL)** and therefore unsupported / a standing operational and human-
time-cost risk.

> *Across the catalogued solution base, is the obsolete (EOL) share large enough that replacing the
> physical PBX is **necessary** rather than optional?*

It reuses the **same original data** as notebook 06 (`analyze_registry` / `solution_registry.csv`).
It shares **no metric** with notebook 10 (financial NPV) or notebook 12 (cybersecurity/integration),
so the four feasibility legs stay disjoint: financial → NB 10, technical/lifecycle → **here**,
cybersecurity & integration → NB 12.

**繁中.** 本筆記本套用**虛無假設顯著性檢定**，回答筆記本 10（財務）未涵蓋的上游問題：
**汰換實體 PBX 是否*必要*？** 驅動因素為**技術過時**——已達**生命終止（EOL）**、不再受支援而
構成營運與人力時間成本風險的安裝基數比例。沿用與筆記本 06 相同的原始資料，與 NB 10、NB 12 不共用任何指標。

## 1. Hypothesis design / 假設設計

**EN.** We treat a solution as *must-replace* when its lifecycle is **end-of-life**. Per the whole
catalogued base we test:

- **H₀ (null):** replacing is **not** necessary — the EOL share `p_eol ≤ 0.5` (base is predominantly supported).
- **H₁ (alternative):** replacing **is** necessary — `p_eol > 0.5` (a majority is obsolete).
- **Test:** one-sided **exact binomial test** on `lifecycle_assigned == 'most_used_eol'`.
- **Decision rule:** reject H₀ at **α = 0.05**; we also report the **95% Wilson CI** on `p_eol`.

**繁中.** 將生命週期為 **EOL** 的方案視為*須汰換*；對整個目錄基數檢定 H₀：`p_eol ≤ 0.5`，
對立 H₁：`p_eol > 0.5`。採單尾**精確二項檢定**，α = 0.05，並報告 `p_eol` 的 95% Wilson 信賴區間。

In [ ]:
# Run from the repository root (same convention as notebooks 06/10).
import sys
import json
import pandas as pd
from pathlib import Path
from scipy import stats

# Resolve the repo root whether launched from root (Jupyter) or notebooks/ (nbconvert).
ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.research.product_researcher import analyze_registry

# --- Original data, identical source to notebook 06 ---
reg_path = ROOT / "data" / "processed" / "solution_registry.csv"
if reg_path.exists():
    df = pd.read_csv(reg_path)
    print(f"Registry loaded from CSV: {len(df)} solutions")
else:
    df = analyze_registry()
    print(f"Registry generated via analyze_registry: {len(df)} solutions")

In [ ]:
ALPHA = 0.05
NULL_P = 0.50  # H0: EOL share <= 50%

is_eol = (df["lifecycle_assigned"] == "most_used_eol")
n = int(len(df))
k = int(is_eol.sum())
p_hat = k / n if n else float("nan")

# One-sided exact binomial test: H1 is p_eol > 0.5 (greater)
bt = stats.binomtest(k, n, NULL_P, alternative="greater")
ci = bt.proportion_ci(confidence_level=1 - ALPHA, method="wilson")
reject_H0 = bool(bt.pvalue < ALPHA)

print("=" * 60)
print("  11  NECESSITY (technical / lifecycle) — Null-Hypothesis Test")
print("=" * 60)
print(f"  Solutions evaluated (n):         {n}")
print(f"  End-of-life (EOL) solutions (k): {k}")
print(f"  EOL share (p_hat):               {p_hat:.3f}")
print(f"  H0: p_eol <= {NULL_P:.2f}   H1: p_eol > {NULL_P:.2f}")
print(f"  One-sided binomial p-value:      {bt.pvalue:.4g}")
print(f"  {int((1 - ALPHA) * 100)}% CI on p_eol (Wilson):     [{ci.low:.3f}, {ci.high:.3f}]")
print("-" * 60)
if reject_H0:
    print(f"  REJECT H0 at alpha={ALPHA}: obsolescence makes replacing the")
    print("  physical PBX a TECHNICAL NECESSITY (majority of base is EOL).")
else:
    print(f"  FAIL TO REJECT H0 at alpha={ALPHA}: lifecycle evidence does NOT")
    print("  establish replacement as technically necessary on its own.")
print("=" * 60)

# Per-category breakdown for the frontend chart
cat_order = ["cutting_edge", "mature_active", "most_used_current", "most_used_eol"]
counts = df["lifecycle_assigned"].value_counts()
lifecycle_counts = {c: int(counts.get(c, 0)) for c in cat_order}
lifecycle_counts

In [ ]:
# Persist verdict CSV (committed back to main by the report workflow)
necessity_lifecycle = pd.DataFrame([{
    "dimension": "technical_lifecycle",
    "metric": "eol_share",
    "n": n, "k_eol": k, "p_hat": p_hat,
    "null_p": NULL_P, "alternative": "greater",
    "p_value": bt.pvalue, "ci_low": ci.low, "ci_high": ci.high,
    "alpha": ALPHA, "reject_H0": reject_H0,
    "verdict": "replacement_necessary" if reject_H0 else "replacement_not_established",
}])
out_csv = ROOT / "data" / "processed" / "replacement_necessity_lifecycle.csv"
necessity_lifecycle.to_csv(out_csv, index=False)
print(f"Saved CSV → {out_csv}")
display(necessity_lifecycle)

In [ ]:
# ---- Export results to the frontend (same pattern as notebooks 09/10) ----
FRONTEND_DATA = ROOT / "frontend" / "data"
FRONTEND_DATA.mkdir(parents=True, exist_ok=True)

payload = {
    "title_en": "Necessity Test — Technical / Lifecycle",
    "title_zh": "必要性檢定—技術／生命週期",
    "method_en": "One-sided exact binomial test on the end-of-life (EOL) share of the catalogued solution base.",
    "method_zh": "對目錄方案基數的生命終止（EOL）比例進行單尾精確二項檢定。",
    "alpha": ALPHA,
    "null_p": NULL_P,
    "n": n,
    "k_eol": k,
    "p_hat": round(p_hat, 4),
    "p_value": float(bt.pvalue),
    "ci_low": round(ci.low, 4),
    "ci_high": round(ci.high, 4),
    "reject_h0": reject_H0,
    "verdict_en": ("Replacement IS technically necessary" if reject_H0
                   else "Replacement not established on lifecycle alone"),
    "verdict_zh": ("汰換具技術必要性" if reject_H0
                   else "僅生命週期不足以證明汰換必要"),
    "lifecycle_counts": lifecycle_counts,
    "sources": [
        {"name": "Hypothesis-testing framework (yongxi-stat)", "url": "https://www.yongxi-stat.com/hypothesis-stat/"},
        {"name": "Solution registry (notebook 06)", "url": "https://github.com/dennislee928/pbx_estimation/blob/main/notebooks/06_product_research.ipynb"},
    ],
}

out_json = FRONTEND_DATA / "lifecycle_necessity.json"
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Frontend JSON → {out_json}")
print(json.dumps(payload, ensure_ascii=False, indent=2)[:600])